```python
import inspect
import json
from copy import deepcopy
from pathlib import Path

import torch
import transformers
from PIL import Image, ImageDraw
from transformers import AutoProcessor

MODEL_PATH = "/data1/models/Qwen3.5-4B"

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True,
)

tokenizer = processor.tokenizer
image_processor = processor.image_processor
```

### 一个基础的流程

- parquet row
```json
{
    "images": [PIL.Image(size=(250, 258))],
    "problem": "<image>Find x.",
    "answer": "3",
}
```
- `RLHFDataset.__getitem__()`
    - _build_messages() 输出
        - `<image>` 被替换为结构化多模态内容：
```python
raw_prompt = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": PIL.Image(size=(250, 258)),
            },
            {
                "type": "text",
                "text": (
                    "Find x. You FIRST think about the reasoning process ..."
                ),
            },
        ],
    }
]
```

```
processor(
    text=[raw_prompt_string],
    images=[pil_image_252x252],
    return_tensors="pt",
)
```

- Qwen2.5-VL-7B
- 对于这一张 $252\times252$ 图：$g_h = 252/14 = 18,\qquad g_w = 252/14 = 18,\qquad g_t = 1$
    - $N_{\text{patch}}= g_tg_hg_w=1\times18\times18=324$
        - raw patch 数量
    - $D_{\text{pixel-patch}}=C\times T\times P\times P=3\times2\times14\times14=1176$
        - 每个 patch 的展平维度
    - 由于 spatial_merge_size=2，每 $2\times2$ 个视觉 patch 最后对应一个 LLM image token：
        - $N_{\text{image-token}}=\frac{18\times18}{2^2}=81$

```
model_inputs = {
    "input_ids": torch.LongTensor,         # [1, L_prompt]
    "attention_mask": torch.LongTensor,    # [1, L_prompt]

    "pixel_values": torch.FloatTensor,     # [324, 1176]
    "image_grid_thw": torch.LongTensor,    # [1, 3], [ [1, 18, 18] ]
}
```

#### qwen3.5

> patch_size = 16, temporal_patch_size = 2, merge_size = 2; factor = patch_size × merge_size = 32

- 原始图片:          640 × 360
- image_grid_thw:   `[1, 22, 40]`（time, height, width）
    - 对齐到32 的倍数 => `(640, 352)`
        - $640 \rightarrow \operatorname{round}(640/32)\times32=640, 360 \rightarrow \operatorname{round}(360/32)\times32=352$
    - 按 16×16 切成 patch
        - $H_{\text{grid}}=352/16=22, W_{\text{grid}}=640/16=40$
- image_pad 数量:   220
    - 视觉编码器首先得到：$1\times22\times40=880$ 个原始视觉 patch。随后 merge_size=2 会将每个 $2\times2$ patch 合并为一个视觉 token：
        - $N_{\text{vision}}=\frac{1\times22\times40}{2^2}=220$
    - +2: `<|vision_start|><|image_pad|><|vision_end|>`
- pixel_values:     `[880, 1536]`
    - 3\*2\*16*16 => 1536

### user text

```python
VALID_ACTIONS = [
    "MoveAhead",
    "MoveBack",
    "MoveLeft",
    "MoveRight",
    "RotateRight",
    "RotateLeft",
    "LookUp",
    "LookDown",
    "Stop",
]

SYSTEM_PROMPT = (
    "You are a navigation agent in an indoor environment. "
    "Your task is to navigate and adjust your viewpoint to PRECISELY match a target image. "
    "You must match the exact position, orientation, and camera angle "
    "— the goal is for your observation to look identical to the target.\n\n"
    "Available actions:\n"
    "- MoveAhead: Move forward 0.25m\n"
    "- MoveBack: Move backward 0.25m\n"
    "- MoveLeft: Move left 0.25m\n"
    "- MoveRight: Move right 0.25m\n"
    "- RotateRight: Rotate clockwise 45°\n"
    "- RotateLeft: Rotate counter-clockwise 45°\n"
    "- LookUp: Tilt camera up 30°\n"
    "- LookDown: Tilt camera down 30°\n"
    "- Stop: Declare that you have reached the target viewpoint\n\n"
    "You will receive:\n"
    "1. Your current observation\n"
    "2. Your recent action history (if available)\n"
    "3. The target viewpoint you need to match\n\n"
    "Use your action history to avoid repeating ineffective actions "
    "(e.g. if MoveAhead caused a collision, try a different direction).\n\n"
    "You MUST respond in exactly this format:\n"
    "Action: <action name>"
)

user_text = "\n".join([
    "Your CURRENT observation:",
    "<image>",
    "TARGET viewpoint you must match:",
    "<image>",
    f"Valid actions at this step: {', '.join(VALID_ACTIONS)}",
])

print(user_text)
```

----

```
Your CURRENT observation:
<image>
TARGET viewpoint you must match:
<image>
Valid actions at this step: MoveAhead, MoveBack, MoveLeft, MoveRight, RotateRight, RotateLeft, LookUp, LookDown, Stop
```

### messages

```python
segments = user_text.split("<image>")

content = []
for index, segment in enumerate(segments):
    if segment:
        content.append({"type": "text", "text": segment})
    if index < len(segments) - 1:
        content.append({"type": "image"})

assert sum(block["type"] == "image" for block in content) == 2

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": content},
]

print(json.dumps(messages, ensure_ascii=False, indent=2))
```

----

```
[
  {
    "role": "system",
    "content": "You are a navigation agent in an indoor environment. ..."
  },
  {
    "role": "user",
    "content": [
      {
        "type": "text",
        "text": "Your CURRENT observation:\n"
      },
      {
        "type": "image"
      },
      {
        "type": "text",
        "text": "\nTARGET viewpoint you must match:\n"
      },
      {
        "type": "image"
      },
      {
        "type": "text",
        "text": "\nValid actions at this step: MoveAhead, MoveBack, MoveLeft, MoveRight, RotateRight, RotateLeft, LookUp, LookDown, Stop"
      }
    ]
  }
]
```

### image input

```python
current = Image.new("RGB", (640, 360), (38, 87, 146))
target = Image.new("RGB", (640, 360), (151, 74, 57))

ImageDraw.Draw(current).text(
    (20, 20),
    "CURRENT 640x360",
    fill="white",
)

ImageDraw.Draw(target).text(
    (20, 20),
    "TARGET 640x360",
    fill="white",
)

images = [current, target]

print([
    ("current", current.size),
    ("target", target.size),
])
```

### prompt

```python
inline_content = deepcopy(content)

image_block_indices = [
    index
    for index, block in enumerate(inline_content)
    if block["type"] == "image"
]

inline_content[image_block_indices[0]]["image"] = current
inline_content[image_block_indices[1]]["image"] = target

inline_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": inline_content},
]
one_call_inputs = processor.apply_chat_template(
    inline_messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
)
print(tokenizer.decode(one_call_inputs['input_ids'])[0])
```

------
```
<|im_start|>system
You are a navigation agent in an indoor environment. ...<|im_end|>
<|im_start|>user
Your CURRENT observation:
<|vision_start|><|image_pad|>*220<|vision_end|>
TARGET viewpoint you must match:
<|vision_start|><|image_pad|>*220<|vision_end|>
Valid actions at this step: MoveAhead, MoveBack, MoveLeft, MoveRight, RotateRight, RotateLeft, LookUp, LookDown, Stop<|im_end|>
<|im_start|>assistant
<think>

</think>
```

### raw prompt

```python
raw_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(raw_prompt)
```
----
```
<|im_start|>system
You are a navigation agent in an indoor environment...<|im_end|>
<|im_start|>user
Your CURRENT observation:
<|vision_start|><|image_pad|><|vision_end|>
TARGET viewpoint you must match:
<|vision_start|><|image_pad|><|vision_end|>
Valid actions at this step: MoveAhead, MoveBack, MoveLeft, MoveRight, RotateRight, RotateLeft, LookUp, LookDown, Stop<|im_end|>
<|im_start|>assistant
<think>

</think>
```

### tokenize 和视觉预处理

```python
model_inputs = processor(
    text=[raw_prompt],
    images=images,
    return_tensors="pt",
)

for name, value in model_inputs.items():
    if isinstance(value, torch.Tensor):
        print(
            f"{name:20s}",
            f"shape={tuple(value.shape)}",
            f"dtype={value.dtype}",
        )
    else:
        print(name, type(value).__name__)
```

----

```
input_ids            shape=(1, 753) dtype=torch.int64
attention_mask       shape=(1, 753) dtype=torch.int64
mm_token_type_ids    shape=(1, 753) dtype=torch.int64
pixel_values         shape=(1760, 1536) dtype=torch.float32
image_grid_thw       shape=(2, 3) dtype=torch.int64
```

-----

```python
input_ids = model_inputs["input_ids"][0]

special_token_ids = {
    "vision_start": tokenizer.convert_tokens_to_ids("<|vision_start|>"),
    "image_pad": tokenizer.convert_tokens_to_ids("<|image_pad|>"),
    "vision_end": tokenizer.convert_tokens_to_ids("<|vision_end|>"),
}

print(special_token_ids)

special_token_counts = {
    name: int((input_ids == token_id).sum())
    for name, token_id in special_token_ids.items()
}

print(special_token_counts)
```

-------

```json
{
    'vision_start': 248053,
    'image_pad': 248056,
    'vision_end': 248054,
}

{
    'vision_start': 2,
    'image_pad': 440,
    'vision_end': 2,
}
```